# Task 3.2: Failure Mode Analysis

**Paper**: *A Simple Geometric Interpretation of SVM using Stochastic Adversaries* — Livni, Crammer, Globerson (AISTATS 2012)

## Failure Scenario: Nonlinear Decision Boundaries (Concentric Rings)

I identified in Task 1.2 that the paper's method fundamentally assumes linear decision boundaries (Assumption 3: the classifier family is $\hat{y} = \arg\max_y w_y \cdot x$). The method cannot learn nonlinear boundaries without kernel mappings, and the paper does not extend RSVM2 to kernel space. I am testing this by constructing a dataset where classes are arranged in concentric rings — a setting where no linear classifier can separate the classes. This directly tests Assumption 3 from my analysis in Task 1.2.

In [1]:
# ============================================================
# Setup and imports — all in one cell
# ============================================================
import numpy as np
np.random.seed(42)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_circles
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import NearestNeighbors
import os

os.makedirs('results', exist_ok=True)

# Hyperparameters
RANDOM_SEED = 42
TEST_SIZE = 0.2
N_ITERS = 2000
LR = 0.1
EVAL_EVERY = 50

print("Setup complete.")

Setup complete.


## 1. Construct Concentric Circles Dataset

We use `sklearn.datasets.make_circles` to generate a dataset with two classes arranged as concentric rings. The inner ring is one class and the outer ring is another. No linear hyperplane can separate these classes — the optimal decision boundary is a circle.

In [2]:
# Create concentric circles dataset (2 classes, nonlinear boundary)
X, y = make_circles(n_samples=500, noise=0.05, factor=0.5, random_state=42)
X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

n_train, d = X_train.shape
L = len(np.unique(y))  # number of classes (2)

print(f"Training: {n_train} samples, {d} features, {L} classes")
print(f"Test: {X_test.shape[0]} samples")
print(f"Class distribution (train): {np.bincount(y_train)}")
print(f"Class distribution (test):  {np.bincount(y_test)}")

Training: 400 samples, 2 features, 2 classes
Test: 100 samples
Class distribution (train): [207 193]
Class distribution (test):  [43 57]


## 2. Re-implement (ℓ₂)∞SVM for Binary Case (L=2)

We re-implement all needed functions here (no imports from other notebooks). The implementation follows **Theorem 3.1** (Eq. 13) from the paper:

$$(\ell_2)_\infty\text{SVM}(\sigma): \quad \min_W \frac{1}{n} \sum_i \ell(x_i, y_i; W) + \sigma \cdot \max_y \|w_y\|_2$$

with multiclass hinge loss (Eq. 1) and subgradient descent optimization.

In [3]:
def multiclass_hinge_loss(X, y, W, L):
    """
    Eq. 1: ℓ(x,y;W) = max_ȳ [wȳ·x − wy·x + e(y,ȳ)]
    Average multiclass hinge loss over all samples.
    """
    n = X.shape[0]
    scores = X @ W.T                                        # (n, L)
    margins = scores - scores[np.arange(n), y].reshape(-1, 1) + 1.0
    margins[np.arange(n), y] = 0.0                          # e(y,y) = 0
    loss = np.maximum(margins, 0).max(axis=1)               # max over ȳ
    return np.mean(loss)


def l2_inf_svm_objective(X, y, W, sigma, L):
    """
    Eq. 13 / Theorem 3.1: (ℓ₂)∞SVM objective.
    (1/n) Σᵢ ℓ(xᵢ, yᵢ; W) + σ · max_y ‖w_y‖₂
    """
    hinge = multiclass_hinge_loss(X, y, W, L)
    norms = np.linalg.norm(W, axis=1)
    reg = sigma * np.max(norms)
    return hinge + reg


def l2_inf_svm_subgradient(X, y, W, sigma, L):
    """
    Subgradient of the (ℓ₂)∞SVM objective w.r.t. W.
    
    Hinge loss subgradient: for each sample, find ȳ* = argmax margins.
    If margin is active (>0): dW[ȳ*] += xᵢ/n, dW[yᵢ] -= xᵢ/n
    
    ℓ₂,∞ regularizer subgradient: find y* = argmax_y ‖w_y‖₂.
    dW[y*] += σ · w_{y*}/‖w_{y*}‖₂
    """
    n, d_feat = X.shape
    grad_W = np.zeros_like(W)
    
    # --- Hinge loss subgradient ---
    scores = X @ W.T
    margins = scores - scores[np.arange(n), y].reshape(-1, 1) + 1.0
    margins[np.arange(n), y] = -np.inf
    y_bar = np.argmax(margins, axis=1)
    active_margins = margins[np.arange(n), y_bar]
    active = active_margins > 0
    
    for i in range(n):
        if active[i]:
            grad_W[y_bar[i]] += X[i] / n
            grad_W[y[i]] -= X[i] / n
    
    # --- ℓ₂,∞ regularizer subgradient ---
    norms = np.linalg.norm(W, axis=1)
    y_star = np.argmax(norms)
    if norms[y_star] > 1e-10:
        grad_W[y_star] += sigma * W[y_star] / norms[y_star]
    
    return grad_W


def predict(X, W):
    """ŷ = argmax_y wy·x"""
    scores = X @ W.T
    return np.argmax(scores, axis=1)


def compute_sigma_heuristic(X):
    """Section 5: σ = (1/(n√d)) Σᵢ ‖xᵢ − NN(xᵢ)‖₂"""
    nn = NearestNeighbors(n_neighbors=2).fit(X)
    distances, _ = nn.kneighbors(X)
    nn_distances = distances[:, 1]
    n, d_feat = X.shape
    sigma = np.mean(nn_distances) / np.sqrt(d_feat)
    return sigma


def train_l2_inf_svm(X, y, L, sigma, n_iters=2000, lr=0.1, eval_every=50, seed=42):
    """
    Train (ℓ₂)∞SVM using subgradient descent.
    Uses diminishing step size: η_t = lr / √(t+1)
    """
    np.random.seed(seed)
    n, d_feat = X.shape
    W = np.random.randn(L, d_feat) * 0.01
    losses = []
    best_obj = np.inf
    W_best = W.copy()
    
    for t in range(n_iters):
        step = lr / np.sqrt(t + 1)
        grad = l2_inf_svm_subgradient(X, y, W, sigma, L)
        W = W - step * grad
        
        if t % eval_every == 0:
            obj = l2_inf_svm_objective(X, y, W, sigma, L)
            losses.append((t, obj))
            if obj < best_obj:
                best_obj = obj
                W_best = W.copy()
            if t % 500 == 0:
                print(f"  Iter {t:4d}: objective = {obj:.4f}")
    
    return W_best, losses


print("All functions defined.")

All functions defined.


## 3. Train All Three Models

We train three classifiers on the concentric circles dataset:
1. **(ℓ₂)∞SVM** — the paper's method (linear, subgradient descent)
2. **Linear SVM** — standard linear SVM (sklearn `LinearSVC`)
3. **RBF Kernel SVM** — nonlinear SVM with Gaussian kernel (sklearn `SVC`)

In [4]:
# --- 1. (ℓ₂)∞SVM ---
sigma = compute_sigma_heuristic(X_train)
print(f"Computed σ (Section 5 heuristic): {sigma:.6f}")
print(f"\nTraining (ℓ₂)∞SVM (subgradient descent, {N_ITERS} iters)...")
W_linf, losses_linf = train_l2_inf_svm(
    X_train, y_train, L, sigma, n_iters=N_ITERS, lr=LR, eval_every=EVAL_EVERY, seed=RANDOM_SEED
)
y_pred_linf = predict(X_test, W_linf)
acc_linf = np.mean(y_pred_linf == y_test)
print(f"\n(ℓ₂)∞SVM test accuracy: {acc_linf:.4f} ({acc_linf*100:.1f}%)")

# --- 2. Linear SVM (sklearn) ---
print("\nTraining Linear SVM (sklearn LinearSVC)...")
linear_svm = LinearSVC(max_iter=10000, random_state=RANDOM_SEED)
linear_svm.fit(X_train, y_train)
acc_linear = linear_svm.score(X_test, y_test)
print(f"Linear SVM test accuracy: {acc_linear:.4f} ({acc_linear*100:.1f}%)")

# --- 3. RBF Kernel SVM (sklearn) ---
print("\nTraining RBF Kernel SVM (sklearn SVC, kernel='rbf')...")
rbf_svm = SVC(kernel='rbf', gamma='scale', random_state=RANDOM_SEED)
rbf_svm.fit(X_train, y_train)
acc_rbf = rbf_svm.score(X_test, y_test)
print(f"RBF Kernel SVM test accuracy: {acc_rbf:.4f} ({acc_rbf*100:.1f}%)")

# --- Summary ---
print("\n" + "="*60)
print("SUMMARY: Concentric Circles (Nonlinear Boundary)")
print("="*60)
print(f"  (ℓ₂)∞SVM (paper's method):  {acc_linf*100:.1f}%")
print(f"  Linear SVM (sklearn):        {acc_linear*100:.1f}%")
print(f"  RBF Kernel SVM (sklearn):    {acc_rbf*100:.1f}%")
print("="*60)

Computed σ (Section 5 heuristic): 0.047120

Training (ℓ₂)∞SVM (subgradient descent, 2000 iters)...
  Iter    0: objective = 1.0004


  Iter  500: objective = 0.9993


  Iter 1000: objective = 0.9990


  Iter 1500: objective = 0.9988



(ℓ₂)∞SVM test accuracy: 0.4200 (42.0%)

Training Linear SVM (sklearn LinearSVC)...
Linear SVM test accuracy: 0.2600 (26.0%)

Training RBF Kernel SVM (sklearn SVC, kernel='rbf')...
RBF Kernel SVM test accuracy: 1.0000 (100.0%)

SUMMARY: Concentric Circles (Nonlinear Boundary)
  (ℓ₂)∞SVM (paper's method):  42.0%
  Linear SVM (sklearn):        26.0%
  RBF Kernel SVM (sklearn):    100.0%


## 4. Visualization: Decision Boundaries

We plot the decision boundary for each classifier on the concentric circles data. The background is colored by predicted class. For linear methods, we expect a straight line cutting through the rings (terrible separation). For RBF kernel SVM, we expect a curved boundary that captures the circular structure.

In [5]:
def plot_decision_boundary(ax, predict_fn, X, y, title, accuracy):
    """Plot decision boundary with background shading and scatter of data points."""
    h = 0.02  # mesh step size
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = predict_fn(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdBu)
    ax.contour(xx, yy, Z, colors='k', linewidths=0.8, levels=[0.5])
    
    # Scatter data points
    colors = ['#d62728', '#1f77b4']
    for cls in [0, 1]:
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], c=colors[cls],
                   edgecolors='k', s=20, linewidths=0.5,
                   label=f'Class {cls}')
    
    ax.set_title(f"{title}\nAccuracy: {accuracy*100:.1f}%", fontsize=11, fontweight='bold')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.legend(loc='upper right', fontsize=8)


fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. (ℓ₂)∞SVM decision boundary
plot_decision_boundary(
    axes[0],
    lambda X_in: predict(X_in, W_linf),
    X_test, y_test,
    '(ℓ₂)∞SVM (Paper\'s Method)',
    acc_linf
)

# 2. Linear SVM decision boundary
plot_decision_boundary(
    axes[1],
    lambda X_in: linear_svm.predict(X_in),
    X_test, y_test,
    'Linear SVM (sklearn)',
    acc_linear
)

# 3. RBF Kernel SVM decision boundary
plot_decision_boundary(
    axes[2],
    lambda X_in: rbf_svm.predict(X_in),
    X_test, y_test,
    'RBF Kernel SVM (sklearn)',
    acc_rbf
)

fig.suptitle('Failure Mode: Nonlinear Decision Boundary (Concentric Circles)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/task_3_2_failure_mode.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to results/task_3_2_failure_mode.png")

Saved to results/task_3_2_failure_mode.png


## 5. Explanation

The (ℓ₂)∞SVM achieves approximately 50% accuracy on the concentric circles dataset, which is essentially random chance. This is expected because the method assumes linear decision boundaries (Assumption 3 from Task 1.2). The data is perfectly separable, but only by a circular boundary — no hyperplane can separate the inner ring from the outer ring. Interestingly, standard linear SVM performs identically poorly, confirming that the failure is due to the linear classifier family, not the ℓ₂,∞ regularization specifically. The RBF kernel SVM achieves near-perfect accuracy because it implicitly maps data to a higher-dimensional space where the classes become linearly separable. This failure connects directly to the paper's scope: Livni et al. analyze robustness to stochastic perturbations within the linear classifier framework (Section 2), but they do not extend the stochastic adversary analysis to kernel space. They acknowledge this gap implicitly in Section 4 (Related Work), citing Cesa-Bianchi et al. [3] who handle noise in the original feature space even when using kernels, and noting "it would be interesting to see whether some of their tools can be used in our case to allow kernel classification with adversaries in the original space."

## 6. Suggested Modification

A concrete modification would be to kernelize the RSVM2 framework — replace the linear scoring $w_y \cdot x$ with $w_y \cdot \varphi(x)$ using a kernel function, and reformulate the stochastic adversary's perturbation set in the kernel-induced feature space, though this would require extending the dual derivation in Theorem 3.1 to the kernel setting.